# ModelID m_011 — KNN Classifier


## 1) Metadata and Imports
**Content:** Define ModelID, runtime metadata, canonical paths, and imports for a K-nearest-neighbors experiment.


In [1]:
from pathlib import Path
import importlib.util
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

MODEL_ID = "m_011"
RUNNER = "codex"
MACHINE = "cpu"
DATA_SCOPE = "full"
RANDOM_STATE = 42

cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == "train_nb" else cwd / "model_training"
assert BASE_DIR.name == "model_training", f"Unexpected BASE_DIR: {BASE_DIR}"

TRAIN_FACTOR = BASE_DIR / "training_data" / "train_df_factor.csv"
TRAIN_TARGET = BASE_DIR / "training_data" / "train_df_target.csv"
VAL_FACTOR = BASE_DIR / "val_data" / "val_df_factor.csv"
VAL_TARGET = BASE_DIR / "val_data" / "val_df_target.csv"
SCORING_SCRIPT = BASE_DIR / "help_stuff" / "validation_score.py"
MODEL_PATH = BASE_DIR / "trained_models" / f"{MODEL_ID}_model.pkl"

metadata = {
    "model_id": MODEL_ID,
    "runner": RUNNER,
    "machine": MACHINE,
    "data_scope": DATA_SCOPE,
    "random_state": RANDOM_STATE,
}
metadata


{'model_id': 'm_011',
 'runner': 'codex',
 'machine': 'cpu',
 'data_scope': 'full',
 'random_state': 42}

## 2) Data Loading
**Content:** Load training factors, training target, validation factors, and validation target from canonical CSV files.


In [2]:
def load_data():
    X_train = pd.read_csv(TRAIN_FACTOR, low_memory=False)
    y_train = pd.read_csv(TRAIN_TARGET)["Prepaied_3m"].astype(int)
    X_val = pd.read_csv(VAL_FACTOR, low_memory=False)
    y_val = pd.read_csv(VAL_TARGET)["Prepaied_3m"].astype(int)
    return X_train, y_train, X_val, y_val

X_train_raw, y_train, X_val_raw, y_val = load_data()

{
    "train_factor_shape": X_train_raw.shape,
    "train_target_shape": y_train.shape,
    "val_factor_shape": X_val_raw.shape,
    "val_target_shape": y_val.shape,
}


{'train_factor_shape': (45062, 74),
 'train_target_shape': (45062,),
 'val_factor_shape': (104, 74),
 'val_target_shape': (104,)}

## 3) Required Data Checks
**Content:** Validate row counts, row alignment, target distribution, missingness, duplicate columns, schema mismatch, feature types, and obvious leakage columns before training.


In [3]:
def find_obvious_leakage_columns(columns):
    leakage_terms = ["prepaied_3m", "prepaid_3m", "target", "label"]
    return [c for c in columns if any(term in c.lower() for term in leakage_terms)]


def run_required_checks(X_train, y_train, X_val, y_val):
    numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
    schema_mismatch = sorted(set(X_train.columns).symmetric_difference(set(X_val.columns)))
    checks = {
        "train_factor_shape": X_train.shape,
        "train_target_shape": y_train.shape,
        "val_factor_shape": X_val.shape,
        "val_target_shape": y_val.shape,
        "train_row_alignment": len(X_train) == len(y_train),
        "val_row_alignment": len(X_val) == len(y_val),
        "schema_order_match": list(X_train.columns) == list(X_val.columns),
        "schema_mismatch_count": len(schema_mismatch),
        "duplicate_columns_train": int(X_train.columns.duplicated().sum()),
        "duplicate_columns_val": int(X_val.columns.duplicated().sum()),
        "missing_cells_train": int(X_train.isna().sum().sum()),
        "missing_cells_val": int(X_val.isna().sum().sum()),
        "train_target_distribution": y_train.value_counts().sort_index().to_dict(),
        "val_target_distribution": y_val.value_counts().sort_index().to_dict(),
        "train_positive_rate": float(y_train.mean()),
        "val_positive_rate": float(y_val.mean()),
        "numeric_column_count": len(numeric_cols),
        "categorical_column_count": len(categorical_cols),
        "obvious_leakage_columns": find_obvious_leakage_columns(X_train.columns),
    }
    return checks

checks = run_required_checks(X_train_raw, y_train, X_val_raw, y_val)
assert checks["train_row_alignment"]
assert checks["val_row_alignment"]
assert checks["schema_order_match"]
assert checks["duplicate_columns_train"] == 0
assert checks["duplicate_columns_val"] == 0
assert not checks["obvious_leakage_columns"]
checks


{'train_factor_shape': (45062, 74),
 'train_target_shape': (45062,),
 'val_factor_shape': (104, 74),
 'val_target_shape': (104,),
 'train_row_alignment': True,
 'val_row_alignment': True,
 'schema_order_match': True,
 'schema_mismatch_count': 0,
 'duplicate_columns_train': 0,
 'duplicate_columns_val': 0,
 'missing_cells_train': 1674885,
 'missing_cells_val': 3879,
 'train_target_distribution': {0: 44876, 1: 186},
 'val_target_distribution': {0: 52, 1: 52},
 'train_positive_rate': 0.004127646353912388,
 'val_positive_rate': 0.5,
 'numeric_column_count': 55,
 'categorical_column_count': 19,
 'obvious_leakage_columns': []}

## 4) Preprocessing and Feature Engineering
**Content:** Drop columns that are all-null in the training factors, identify numeric/categorical features, and build a no-leakage preprocessing pipeline fit only on training data.


In [4]:
def drop_all_null_train_columns(X_train, X_val):
    all_null_cols = [col for col in X_train.columns if X_train[col].isna().all()]
    return X_train.drop(columns=all_null_cols), X_val.drop(columns=all_null_cols), all_null_cols


def get_feature_groups(X_train):
    numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]
    return numeric_cols, categorical_cols


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__", keep_empty_features=True)),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

X_train, X_val, dropped_all_null_columns = drop_all_null_train_columns(X_train_raw, X_val_raw)
numeric_cols, categorical_cols = get_feature_groups(X_train)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

feature_summary = {
    "dropped_all_null_columns": dropped_all_null_columns,
    "numeric_column_count": len(numeric_cols),
    "categorical_column_count": len(categorical_cols),
    "modeling_train_shape": X_train.shape,
    "modeling_val_shape": X_val.shape,
}
feature_summary


{'dropped_all_null_columns': ['UPB at Issuance',
  'Interest Only First Principal And Interest Payment Date',
  'Months to Amortization',
  'Mortgage Insurance Cancellation Indicator',
  'Scheduled Principal Current',
  'Property Preservation and Repair Costs',
  'Asset Recovery Costs',
  'Miscellaneous Holding Expenses and Credits',
  'Associated Taxes for Holding Property',
  'Modification-Related Non-Interest Bearing UPB',
  'Principal Forgiveness Amount',
  'Original List Start Date',
  'Original List Price',
  'Current List Price',
  'Borrower Credit Score At Issuance',
  'Co-Borrower Credit Score At Issuance',
  'Borrower Credit Score Current ',
  'Co-Borrower Credit Score Current',
  'Loan Holdback Effective Date',
  'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator',
  'ARM Product Type',
  'Initial Fixed-Rate Period ',
  'Interest Rate Adjustment Frequency',
  'Next Interest Rate Adjustment Date',
  'Next Payment Change Date',
  'ARM Cap Structure',
  'Initial Interest Rate Cap

## 5) Balanced Training Subset
**Content:** Create a deterministic class-balanced training subset using all positive loans and an equal number of randomly sampled negative loans so KNN is not dominated by the majority class.


In [5]:
def make_balanced_training_subset(X_train, y_train, random_state=42, negative_to_positive_ratio=1):
    rng = np.random.default_rng(random_state)
    positive_idx = np.flatnonzero(y_train.to_numpy() == 1)
    negative_idx = np.flatnonzero(y_train.to_numpy() == 0)
    n_negative = min(len(negative_idx), len(positive_idx) * negative_to_positive_ratio)
    sampled_negative_idx = rng.choice(negative_idx, size=n_negative, replace=False)
    selected_idx = np.concatenate([positive_idx, sampled_negative_idx])
    rng.shuffle(selected_idx)
    return X_train.iloc[selected_idx].copy(), y_train.iloc[selected_idx].copy(), selected_idx

NEGATIVE_TO_POSITIVE_RATIO = 1
X_train_balanced, y_train_balanced, balanced_idx = make_balanced_training_subset(
    X_train, y_train, random_state=RANDOM_STATE, negative_to_positive_ratio=NEGATIVE_TO_POSITIVE_RATIO
)

balanced_subset_summary = {
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "balanced_train_shape": X_train_balanced.shape,
    "balanced_target_distribution": y_train_balanced.value_counts().sort_index().to_dict(),
}
balanced_subset_summary


{'negative_to_positive_ratio': 1,
 'balanced_train_shape': (372, 40),
 'balanced_target_distribution': {0: 186, 1: 186}}

## 6) KNN Pipeline and Cross-Validation
**Content:** Train a KNeighborsClassifier pipeline and select KNN hyperparameters with stratified cross-validation on the balanced training subset only.


In [6]:
def build_knn_pipeline(preprocessor):
    return Pipeline([
        ("preprocess", preprocessor),
        ("knn", KNeighborsClassifier()),
    ])


def run_grid_search(pipeline, X_train, y_train):
    param_grid = {
        "knn__n_neighbors": [3, 5, 7, 11, 15, 21],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2],
        "knn__metric": ["minkowski"],
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        search.fit(X_train, y_train)
    return search

knn_pipeline = build_knn_pipeline(preprocessor)
grid_search = run_grid_search(knn_pipeline, X_train_balanced, y_train_balanced)
best_model = grid_search.best_estimator_

cv_results = pd.DataFrame(grid_search.cv_results_).sort_values("rank_test_score")
search_summary = {
    "best_cv_balanced_accuracy": float(grid_search.best_score_),
    "best_params": grid_search.best_params_,
    "top_cv_rows": cv_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(5).to_dict("records"),
}
search_summary


{'best_cv_balanced_accuracy': 0.75,
 'best_params': {'knn__metric': 'minkowski',
  'knn__n_neighbors': 11,
  'knn__p': 1,
  'knn__weights': 'distance'},
 'top_cv_rows': [{'rank_test_score': 1,
   'mean_test_score': 0.75,
   'std_test_score': 0.03017465634495111,
   'mean_train_score': 1.0,
   'params': {'knn__metric': 'minkowski',
    'knn__n_neighbors': 11,
    'knn__p': 1,
    'knn__weights': 'distance'}},
  {'rank_test_score': 2,
   'mean_test_score': 0.7446236559139785,
   'std_test_score': 0.03314201076865038,
   'mean_train_score': 0.7634408602150536,
   'params': {'knn__metric': 'minkowski',
    'knn__n_neighbors': 11,
    'knn__p': 1,
    'knn__weights': 'uniform'}},
  {'rank_test_score': 3,
   'mean_test_score': 0.7365591397849464,
   'std_test_score': 0.0639536411471522,
   'mean_train_score': 1.0,
   'params': {'knn__metric': 'minkowski',
    'knn__n_neighbors': 21,
    'knn__p': 1,
    'knn__weights': 'distance'}},
  {'rank_test_score': 4,
   'mean_test_score': 0.7311827956

## 7) Validation Prediction and Official Score
**Content:** Predict the validation labels and compute the validation score using the official `help_stuff/validation_score.py` script.


In [7]:
def load_official_scorer(scoring_script):
    spec = importlib.util.spec_from_file_location("validation_score", scoring_script)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.prediction_score

prediction_score = load_official_scorer(SCORING_SCRIPT)
val_pred = best_model.predict(X_val).astype(int)
validation_score = float(prediction_score(val_pred))

validation_summary = {
    "official_validation_score": validation_score,
    "accuracy_score": float(accuracy_score(y_val, val_pred)),
    "balanced_accuracy_score": float(balanced_accuracy_score(y_val, val_pred)),
    "confusion_matrix": confusion_matrix(y_val, val_pred).tolist(),
    "validation_prediction_distribution": pd.Series(val_pred).value_counts().sort_index().to_dict(),
}
validation_summary


{'official_validation_score': 0.6634615384615384,
 'accuracy_score': 0.6634615384615384,
 'balanced_accuracy_score': 0.6634615384615384,
 'confusion_matrix': [[30, 22], [13, 39]],
 'validation_prediction_distribution': {0: 43, 1: 61}}

## 8) Artifact Handling
**Content:** Check model artifact size in memory, but do not persist binary model or validation prediction artifacts because the project PR artifact policy says not to commit generated binaries/CSVs unless explicitly requested.


In [8]:
model_pickle_size_bytes = len(pickle.dumps(best_model))
artifact_summary = {
    "model_path_if_saved": str(MODEL_PATH),
    "model_pickle_size_bytes": model_pickle_size_bytes,
    "model_pickle_size_mb": model_pickle_size_bytes / (1024 * 1024),
    "model_saved": False,
    "validation_prediction_file_generated": False,
    "reason": "Binary model and generated prediction CSV intentionally omitted per PR artifact policy.",
}
artifact_summary


{'model_path_if_saved': '/workspace/Co_AMAP/Co_ML/Experiment_1/model_training/trained_models/m_011_model.pkl',
 'model_pickle_size_bytes': 336649,
 'model_pickle_size_mb': 0.32105350494384766,
 'model_saved': False,
 'validation_prediction_file_generated': False,
 'reason': 'Binary model and generated prediction CSV intentionally omitted per PR artifact policy.'}

## 9) Final Experiment Summary
**Content:** Print the key metadata, selected hyperparameters, validation score, and training-log fields for review.


In [9]:
final_summary = {
    **metadata,
    "model_type": "KNeighborsClassifier with balanced training subset",
    "best_params": grid_search.best_params_,
    "best_cv_balanced_accuracy": float(grid_search.best_score_),
    "official_validation_score": validation_score,
    "model_saved": False,
    "validation_prediction_file_generated": False,
}
final_summary


{'model_id': 'm_011',
 'runner': 'codex',
 'machine': 'cpu',
 'data_scope': 'full',
 'random_state': 42,
 'model_type': 'KNeighborsClassifier with balanced training subset',
 'best_params': {'knn__metric': 'minkowski',
  'knn__n_neighbors': 11,
  'knn__p': 1,
  'knn__weights': 'distance'},
 'best_cv_balanced_accuracy': 0.75,
 'official_validation_score': 0.6634615384615384,
 'model_saved': False,
 'validation_prediction_file_generated': False}